# Step 10: Combine gender from PKG forenames and recovered forenames

Gender per author (AID) is taken from two sources, both inferred with Wiki-Gendersort:

1. **PKG forename** (step 7): majority over all papers of the author; F, M or UNI are final;
2. **recovered forename** (steps 8–9) for all other authors (PKG code `INI`/`UNK` or no forename): again majority over all
   papers; authors whose recovered names give as many F as M votes are set to `UNI` (conflict) and excluded from the analysis;
3. authors without any recovered forename keep their PKG code.

`gender_source` = `pkg` (F/M from the PKG forename) or `recovered` (F/M from the recovered forename).

**Inputs:** `data/derived/filtered_data.csv` (step 5), `data/derived/gender_pkg_by_aid.csv` (step 7),
`data/derived/nan_ini_medline_names_and_gender.csv` (step 9)
**Outputs:** `data/derived/filtered_data_recovered.csv` (columns `AID, PMID, gender, begin_year, topics_num, PubYear`) – input for
`step11_cox_dataprep.ipynb`; `data/derived/gender_by_aid.csv` (columns `AID, gender_pkg, gender_recovered, gender, gender_source`).

In [ ]:
import os
import duckdb

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
DERIVED_DIR = f'{BASE_DIR}/data/derived'
FILTERED_CSV = f'{DERIVED_DIR}/filtered_data.csv'                      # step 5
PKG_GENDER_CSV = f'{DERIVED_DIR}/gender_pkg_by_aid.csv'                # step 7
RECOVERED_CSV = f'{DERIVED_DIR}/nan_ini_medline_names_and_gender.csv'  # step 9
OUT_CSV = f'{DERIVED_DIR}/filtered_data_recovered.csv'
OUT_AID_CSV = f'{DERIVED_DIR}/gender_by_aid.csv'

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")

## 1. Gender per AID from the recovered forenames (majority vote)

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE recovered_gender AS
WITH votes AS (
    SELECT CAST(AID AS BIGINT) AS aid,
           count(*) FILTER (WHERE gender = 'F') AS n_f,
           count(*) FILTER (WHERE gender = 'M') AS n_m,
           mode(gender) FILTER (WHERE gender IN ('UNI', 'INI', 'UNK')) AS other_mode
    FROM read_csv('{RECOVERED_CSV}', header = true, all_varchar = true)
    GROUP BY 1
)
SELECT aid, n_f, n_m,
       CASE WHEN n_f > n_m THEN 'F'
            WHEN n_m > n_f THEN 'M'
            WHEN n_f = n_m AND n_f > 0 THEN 'UNI'      -- conflicting votes
            ELSE other_mode END AS gender_recovered     -- NULL if nothing was recovered -> PKG code is kept below
FROM votes
""")
print(con.execute("""
SELECT gender_recovered, count(*) AS n_authors,
       avg(((n_f > 0) AND (n_m > 0))::INT) AS share_with_conflicting_votes
FROM recovered_gender GROUP BY 1 ORDER BY 2 DESC
""").df())

## 2. Combine both sources and export

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE gender_by_aid AS
SELECT
    CAST(p.AID AS BIGINT) AS aid,
    p.gender_pkg,
    r.gender_recovered,
    CASE WHEN p.gender_pkg IN ('F', 'M', 'UNI') THEN p.gender_pkg
         ELSE coalesce(r.gender_recovered, p.gender_pkg) END AS gender,
    CASE WHEN p.gender_pkg IN ('F', 'M') THEN 'pkg'
         WHEN p.gender_pkg = 'UNI' THEN NULL
         WHEN r.gender_recovered IN ('F', 'M') THEN 'recovered' END AS gender_source
FROM read_csv('{PKG_GENDER_CSV}', header = true, all_varchar = true) p
LEFT JOIN recovered_gender r ON r.aid = CAST(p.AID AS BIGINT)
""")
con.execute(f"""
COPY (SELECT aid AS AID, gender_pkg, gender_recovered, gender, gender_source FROM gender_by_aid)
TO '{OUT_AID_CSV}' (HEADER, DELIMITER ',')
""")
con.execute(f"""
COPY (
    SELECT f.AID, f.PMID, g.gender, f.begin_year, f.topics_num, f.PubYear
    FROM read_csv('{FILTERED_CSV}', header = true, all_varchar = true) f
    LEFT JOIN gender_by_aid g ON g.aid = CAST(f.AID AS BIGINT)        -- authors without any name keep gender = NULL
) TO '{OUT_CSV}' (HEADER, DELIMITER ',')
""")
print('written', OUT_CSV, 'and', OUT_AID_CSV)

In [ ]:
print(con.execute(f"""
SELECT
    gender,
    count(DISTINCT AID) AS n_authors,
    round(100.0 * count(DISTINCT AID) / sum(count(DISTINCT AID)) OVER (), 2) AS pct
FROM read_csv('{OUT_CSV}', header = true, all_varchar = true)
GROUP BY 1
ORDER BY 2 DESC
""").df())

In [ ]:
# authors by source of the gender classification
print(con.execute("""
SELECT gender_source, gender, count(*) AS n_authors
FROM gender_by_aid
GROUP BY ALL
ORDER BY ALL
""").df())